# 🎙️ SLAM-ASR Turkish: Common Voice Benchmark Notebook

Bu notebook, **SLAM-LLM** (Speech-Language Model Framework) metodolojisini birebir uygulayarak **Common Voice Türkçe** veri setinde state-of-the-art (SOTA) ASR başarımını elde etmeniz için tasarlanmıştır.

### 🏗️ Model Mimarisi:
- **Konuşma Kodlayıcı (Speech Encoder):** `openai/whisper-large-v3` (Frozen)
- **Hizalama Katmanı (Projector):** Trainable 2-layer Linear Projector (5x Frame Concatenation Downsampling)
- **Dil Modeli (LLM Backbone):** `Qwen/Qwen2.5-7B-Instruct` veya `meta-llama/Meta-Llama-3-8B-Instruct` (LoRA Fine-tuning)

---

## 1. Donanım & Ekran Kartı Kontrolü (GPU Check)

In [ ]:
!nvidia-smi

## 2. Google Drive Bağlama (Model & Checkpoint Saklama)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Proje dizinini oluştur ve geç
%cd /content
!git clone https://github.com/gokhanersoy/slam-asr.git || true
%cd /content/slam-asr/commonvoice_tr

## 3. Gerekli Kütüphanelerin Kurulumu

In [ ]:
!pip install -q -r requirements.txt

## 4. Türkçe Common Voice Veri Setinin Hazırlanması
Bu adım Hugging Face üzerinden `mozilla-foundation/common_voice_17_0` (Türkçe) veri setini indirir, ses dosyalarını 16kHz mono WAV formatına dönüştürür ve `train.jsonl`, `val.jsonl`, `test.jsonl` dosyalarını oluşturur.

In [ ]:
# Hızlı deneme için varsayılan örnek sayıları verilebilir (Tüm veri seti için --max_train_samples kaldırılabilir)
!python prepare_data.py --output_dir data

## 5. Model Eğitimi (SLAM-ASR Fine-Tuning)
Eğitim sırasında Whisper Large V3 dondurulur, 2 katmanlı Doğrusal İzdüşüm (Linear Projector) katmanı eğitilir ve Qwen2.5-7B-Instruct dil modeline LoRA uygulanır.

In [ ]:
# Eğitimi başlatın
!python train.py train_config.num_epochs=5 train_config.batch_size_training=4 train_config.gradient_accumulation_steps=4

## 6. Test Setinde Değerlendirme (WER & CER Hesaplama)
Eğitilen model test seti üzerinde çalıştırılır, Türkçe karakter ve noktalama normalizasyonunun ardından Word Error Rate (WER) ve Character Error Rate (CER) değerleri hesaplanır.

In [ ]:
!python evaluate.py --checkpoint_path checkpoints/slam_qwen_tr/best_checkpoint.pt --output_file evaluation_results.json

## 7. Örnek Deşifre Playground (Single Audio Inference)
Kendi ses dosyanız veya test setinden rastgele bir ses dosyası üzerinde ASR tahminini test edin.

In [ ]:
import torch
import whisper
from omegaconf import OmegaConf
from src.models.slam_model import model_factory

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = OmegaConf.load("conf/prompt.yaml")

# Modeli yükle
model, tokenizer = model_factory(
    train_config=cfg.train_config,
    model_config=cfg.model_config,
    ckpt_path="checkpoints/slam_qwen_tr/best_checkpoint.pt"
)
model.to(device)
model.eval()

# Örnek Ses Dosyası Yolu
sample_audio_path = "data/audio/test/test_000000.wav"

# Log Mel Spectrogram
audio_raw = whisper.load_audio(sample_audio_path)
audio_raw = whisper.pad_or_trim(audio_raw)
audio_mel = whisper.log_mel_spectrogram(audio_raw, n_mels=128).permute(1, 0).unsqueeze(0).to(device)

# Tahmin Al
prediction = model.generate_transcription(audio_mel=audio_mel)
print(f"🎙️ Tahmin Edilen Metin: {prediction}")
